> 在线讲义：[第 45 章 · 批处理与请求调度：多读一次权重做更多事，但别让别人等太久](https://wikieden.github.io/ai-infra-from-constraints/lessons/45-batching.html) · 本书《从约束推导 AI Infra》

# 第 45 章 · 批处理与请求调度：多读一次权重做更多事，但别让别人等太久

配套讲义：仓库里 `ai-infra/notes/lessons/45-batching.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



每输出 token 的读取量、b_KV、LoRA 容量、连续批处理与分块的步时间、注意力配对取自 AII 第 8 章 §8.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
KiB, MiB, GiB = 2**10, 2**20, 2**30
DW = 15_136_811_008
K = 147_456

**函数 `per_token_read_gib`**：(分摊权重, 本请求 KV) GiB。

In [ ]:
def per_token_read_gib(b: int, L: int) -> tuple[float, float]:
    """(分摊权重, 本请求 KV) GiB。"""
    return DW / b / GiB, L * K / GiB

**函数 `b_kv`**：小工具。

In [ ]:
def b_kv(L: int) -> int:
    return math.ceil(DW / (L * K))

**函数 `lora_mib`**：Q（h→h）与 V（h→kv）各一对低秩矩阵，BF16。

In [ ]:
def lora_mib(rank: int = 16, layers: int = 36, h: int = 4096, kv: int = 1024) -> float:
    """Q（h→h）与 V（h→kv）各一对低秩矩阵，BF16。"""
    q = h * rank + rank * h
    v = h * rank + rank * kv
    return (q + v) * layers * 2 / MiB

**函数 `step_ms`**：实验 8-1 拟合：26.22 ms + 30.4 μs × 新 token + 3 ns × 因果配对。

In [ ]:
def step_ms(new_tokens: int, pairs: int) -> float:
    """实验 8-1 拟合：26.22 ms + 30.4 μs × 新 token + 3 ns × 因果配对。"""
    return 26.22 + 30.4e-3 * new_tokens + 3e-6 * pairs

**函数 `chunk_pairs`**：小工具。

In [ ]:
def chunk_pairs(c: int, h: int) -> int:
    return c * h + c * (c + 1) // 2

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：每输出 token 的读取")
for L in (2048, 8192):
    for b in (1, 16, 64):
        w, kv = per_token_read_gib(b, L)
        print(f"  L = {L}, b = {b:2d}：权重 {w:.2f} + KV {kv:.2f} = {w + kv:.2f} GiB")
    print(f"  L = {L}：b_KV = {b_kv(L)}")
print(f"  2K 下 b 1 → 64 降为 1/{(DW / GiB + 2048 * K / GiB) / (DW / 64 / GiB + 2048 * K / GiB):.0f}")
print(f"  LoRA r = 16（Q、V）：一套 {lora_mib():.1f} MiB，100 套 {100 * lora_mib() / 1024:.2f} GiB")

In [ ]:
print("\n第二步：调度算例的步时间")
print(f"  2K prefill {step_ms(2048, chunk_pairs(2048, 0)):.1f} ms；decode {step_ms(1, 2048):.2f} ms")
print(f"  r2 8K 一次 prefill（+ r0 decode）{step_ms(8192 + 1, chunk_pairs(8192, 0)):.0f} ms")
print(f"  分块 2048：首块 {step_ms(2048, chunk_pairs(2048, 0)):.1f} ms，末块（h = 6144）{step_ms(2048, chunk_pairs(2048, 6144)):.1f} ms")

In [ ]:
print("\n第三步：块内注意力")
a, b = chunk_pairs(512, 0), chunk_pairs(512, 7680)
print(f"  c = 512：首块 {a:,} 对，末块 {b:,} 对（{b / a:.0f} 倍）")

## 3. 代码里才看得到的

1. 步时间模型的三个系数是从 batch 1 实测拟合的：它同时复现 2K prefill 的 94.8 ms 和 decode 的 26.26 ms。
2. decode 一步的"配对"是 1 × 上下文长度：一个新 token 看全部旧 token。
3. LoRA 的 V 投影是 4096 → 1024（8 个 KV 头 × 128），所以 Q 和 V 的低秩矩阵大小不同。

## 4. 自己改一改

- 用 `step_ms` 自己写一个两条请求的调度模拟器，复现 870 / 766 / 844 ms。
- 给 `per_token_read_gib` 加前缀共享：16 条请求共享 6K 时每 token 读多少？
- 把块长从 512 扫到 4096，画末块步时间。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 班车开一趟摊油钱，行李各扛各的；八 K 上下文坐十三人就行李比车重；
> 翻台谁吃完谁走，新客点大菜大家等；
> 读书越往后越要回头对照，块长一样后块慢；
> 电梯多带人门口少等，车里的人到下一层更晚。